# Epsilon Fund - Momentum #2 Portfolio Analysis
Tests how portfolio OOS performance changes depending on portfolio composition and time of trade.

Individual signals and trade logic are **never modified** and depends on each coins input pkl.

---

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import io, contextlib

# ── repo root — works on both Mac and Windows ─────────────────────────────────
ROOT   = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows

WF_DIR = os.path.join(ROOT, 'topics', 'momentum', 'strategies', 'wf_testing_2')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client
from wf_visualizer import plot_portfolio_oos
from wf_engine import cost_stress_test

client = get_binance_client()

---
## Data
Fetch hourly data for all coins with pikl files, and fetch pikl files.



In [130]:
# ── load daily OOS pkl files ───────────────────────────────────────────────────
coin_dfs = {}
for fname in os.listdir(WF_DIR):
    if fname.endswith('_oos.pkl'):
        coin = fname.replace('_oos.pkl', '').upper()
        coin_dfs[coin] = pd.read_pickle(os.path.join(WF_DIR, fname))

print(f'Loaded daily OOS: {list(coin_dfs.keys())}')

Loaded daily OOS: ['ADA', 'XRP', 'ETH', 'BTC', 'SOL', 'BNB', 'AVAX']


In [93]:
# ── fetch 1h data for every coin (one-time pull) ───────────────────────────────
hourly = {}   # coin → 1h DataFrame

for coin, df in coin_dfs.items():
    symbol = coin + 'USDT'
    start  = str(df.index[0].date())
    end    = str(df.index[-1].date())
    klines = client.get_historical_klines(symbol, '1h', start, end)

    h = pd.DataFrame(klines, columns=[
        'Time','Open','High','Low','Close','Volume',
        'Close_time','Quote_volume','Trades','Taker_base','Taker_quote','Ignore'
    ])
    h['Time'] = pd.to_datetime(h['Time'], unit='ms', utc=True)
    for col in ['Open','High','Low','Close']:
        h[col] = h[col].astype(float)
    h = h.set_index('Time')
    hourly[coin] = h
    print(f'  {coin}: {len(h)} hourly bars  ({start} → {end})')

print('\nHourly data ready.')

/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_8899/275522303.py:8: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  ADA: 26281 hourly bars  (2023-04-09 → 2026-04-08)


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_8899/275522303.py:8: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  XRP: 26377 hourly bars  (2023-04-08 → 2026-04-11)


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_8899/275522303.py:8: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



...

  SOL: 22993 hourly bars  (2023-06-27 → 2026-02-09)


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_8899/275522303.py:8: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  BNB: 26281 hourly bars  (2023-04-09 → 2026-04-08)


/var/folders/03/9vtlsl6166nb83fjhd86xdb00000gn/T/ipykernel_8899/275522303.py:8: DeprecationWarning:

Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.



  AVAX: 23017 hourly bars  (2023-08-08 → 2026-03-24)

Hourly data ready.


---
### Scenarios
Re-aligns portfolio's exit/entry prices to a specific hour of the day. Signal generated on "Day T" is executed at the chosen hour on "Day T+1".

| Scenario | Entry | Exit |
|---|---|---|
| `'close'` | Close[T] midnight UTC — baseline (no modification) | Same |
| `integer` | HH:00 UTC on entry day T | HH:00 UTC on day after exit signal T'+1 |



In [140]:
SCENARIO = 1# ← 'close'  |  integer hour 0-23

def apply_scenario(coin_dfs, hourly, scenario):
    if scenario == 'close':
        return coin_dfs

    adjusted = {}
    for coin, df in coin_dfs.items():
        d = df.copy()
        h = hourly[coin].copy()

        # 1. Get the price at the specific hour
        prices_hh = h.loc[h.index.hour == scenario, 'Open'].copy()

        # 2. CRITICAL ALIGNMENT: 
        # If scenario is 0 (00:00), this price is effectively the "Close" 
        # of the PREVIOUS day. We must shift the index back by 1 day 
        # so it aligns with the daily row that generated the signal.
        if isinstance(scenario, int):
            # Move the price of 'Tuesday 00:00' to the 'Monday' row
            prices_hh.index = prices_hh.index.tz_convert(None) - pd.Timedelta(days=1)
            prices_hh.index = prices_hh.index.normalize()

        # 3. Map to the daily dataframe
        prices_hh = prices_hh[~prices_hh.index.duplicated(keep='first')]
        d['Close'] = prices_hh.reindex(d.index).ffill()

        # Remove the 'synthetic_close' / cumprod logic entirely.
        # This prevents "double-calculating" returns if your 
        # downstream code also calculates returns.
        
        adjusted[coin] = d

    return adjusted

coin_dfs_exec = apply_scenario(coin_dfs, hourly, SCENARIO)
label = f'{SCENARIO}h_UTC' if isinstance(SCENARIO, int) else SCENARIO
print(f'\nScenario applied: {label}')




Scenario applied: 1h_UTC


---
## Portfolio Performance
| Parameter | Description |
|---|---|
| `show_coins` | Subset of coins to include in the visual plot. Use `None` to show all|
| `weights` | Manual allocation per coin (e.g., {'BTC': 0.4, ...}). Defaults to Equal-Weight if `None` |
| `benchmark` | `{'BTC': oos_df}` or multi-coin dict for B&H benchmark. `None` = equal-weight B&H of show_coins|


Per coin statistics also printed as table below.

**Cost Stress Test:** re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Fragile strategies collapse; robust ones degrade gradually.

In [141]:
SHOW_COINS = ['BTC','SOL','XRP','ETH',]
#['AVAX','BTC','BNB','SOL','XRP','ETH']   # or list(coin_dfs.keys())

metrics = plot_portfolio_oos(
    coin_dfs   = coin_dfs_exec,
   # weights =  {'ETH': 0.3, 'SOL': 0.2, 'AVAX': 0.,'BTC':0.1,'BNB':0.1,'XRP':0.2},
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},   # benchmark always uses original close
    show       = True,
    cost       = 0.001,
    save_html  = None,
)


# ── per-coin trade statistics table ──────────────────────────────────────────
print(f'\n── Per-Coin Trade Statistics  |  Scenario: {label} ──')
print(f'  {"Coin":<6} {"Trades":>7} {"Win Rate":>10} {"Prof.Factor":>13} {"Avg W/L":>9}')
print(f'  {"─"*6} {"─"*7} {"─"*10} {"─"*13} {"─"*9}')

for coin in SHOW_COINS:
    t = metrics['coin_trade_stats'].get(coin, None)
    if t is None or len(t) == 0:
        print(f'  {coin:<6} {"0":>7} {"—":>10} {"—":>13} {"—":>9}')
        continue
    n      = len(t)
    wins   = t[t['pnl'] > 0]
    losses = t[t['pnl'] < 0]
    wr     = len(wins) / n if n else 0.0
    gp     = wins['pnl'].sum()
    gl     = abs(losses['pnl'].sum())
    pf     = gp / gl if gl > 0 else 0.0
    aw     = wins['pnl'].mean()        if len(wins)   > 0 else 0.0
    al     = abs(losses['pnl'].mean()) if len(losses) > 0 else 0.0
    awl    = aw / al                   if al          > 0 else 0.0
    print(f'  {coin:<6} {n:>7} {wr*100:>9.1f}% {pf:>13.2f} {awl:>9.2f}')

# ── portfolio cost stress test ────────────────────────────────────────────────
BASE_COST        = 0.001
cost_multipliers = (1.0, 1.5, 2.0, 3.0)

print(f"\n{'═'*72}")
print('PORTFOLIO TRANSACTION COST STRESS TEST')
print(f"{'═'*72}")
print(f'{"Cost":>8} {"Mult":>6} {"Sharpe":>8} {"Return":>10} {"MaxDD":>10} {"Calmar":>8} {"PF":>8} {"WR":>8}')
print(f'{"─"*8} {"─"*6} {"─"*8} {"─"*10} {"─"*10} {"─"*8} {"─"*8} {"─"*8}')

for mult in cost_multipliers:
    c = BASE_COST * mult
    with contextlib.redirect_stdout(io.StringIO()):
        m = plot_portfolio_oos(
            coin_dfs   = coin_dfs_exec,
            show_coins = SHOW_COINS,
            benchmark  = {'BTC': coin_dfs['BTC']},
            show       = False,
            cost       = c,
            save_html  = None,
        )
    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0
    print(f'{c:>8.4f} {mult:>5.1f}x {m["sharpe_ratio"]:>8.2f} {m["total_return"]*100:>9.2f}% '
          f'{m["max_drawdown"]*100:>9.2f}% {calmar:>8.2f} {m["profit_factor"]:>8.2f} {m["win_rate"]*100:>7.1f}%')



── Yearly Trade Statistics ──
  Year    Trades   Win Rate   Prof.Factor   Avg W/L
  ────── ─────── ────────── ───────────── ─────────
  2023        71      57.7%          4.46      3.27
  2024        88      56.8%          2.59      1.97
  2025        63      49.2%          2.45      2.53
  2026        11      45.5%          0.89      1.07

── Per-Coin Trade Statistics  |  Scenario: 1h_UTC ──
  Coin    Trades   Win Rate   Prof.Factor   Avg W/L
  ────── ─────── ────────── ───────────── ─────────
  BTC         91      53.8%          2.23      1.91
  SOL         51      60.8%          3.57      2.31
  XRP         46      50.0%          2.50      2.50
  ETH         45      53.3%          3.33      2.91

════════════════════════════════════════════════════════════════════════
PORTFOLIO TRANSACTION COST STRESS TEST
════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF       WR
──────── ────── ──────── ──────